In [1]:
#| include: false
# Disable tqdm progress bars; this is a hidden cell on the docs page
import os

os.environ["TQDM_DISABLE"] = "1"

# Navigate the brain region hierarchy

The 1327 regions in the Allen parcellation are organised in a hierarchical tree. For example, the
region PPY encompasses both the regions PPYd and PPYs. This page finds the regions above and below
a given region, the leaf nodes, and the regions at a given level. For how the tree is organised,
see [The brain region hierarchy](../explanation/brain-region-hierarchy.qmd).

You can browse the hierarchy visually in the [Allen Brain Atlas viewer](https://atlas.brain-map.org/atlas?atlas=602630314#atlas=602630314&plate=576989940&structure=549&x=5280.261048274253&y=3744.2747201492534&zoom=-3&resolution=11.93&z=3).

In [2]:
import numpy as np
from iblatlas.atlas import AllenAtlas

ba = AllenAtlas()
brain_regions = ba.regions

## Ancestors

To find the ancestors of a region, the regions above it in the tree, use `brain_regions.ancestors`.
Here with the region PPYd:

In [3]:
ancestors = brain_regions.ancestors(ids=brain_regions.acronym2id('PPYd'))
ancestors

{'id': array([ 997,    8,  343, 1065,  354,  370, 1069,  185]),
 'name': array(['root', 'Basic cell groups and regions', 'Brain stem', 'Hindbrain',
        'Medulla', 'Medulla motor related', 'Parapyramidal nucleus',
        'Parapyramidal nucleus deep part'], dtype=object),
 'acronym': array(['root', 'grey', 'BS', 'HB', 'MY', 'MY-mot', 'PPY', 'PPYd'],
       dtype=object),
 'rgb': array([[255, 255, 255],
        [191, 218, 227],
        [255, 112, 128],
        [255, 155, 136],
        [255, 155, 205],
        [255, 179, 217],
        [255, 179, 217],
        [255, 179, 217]], dtype=uint8),
 'level': array([0, 1, 2, 3, 4, 5, 6, 7], dtype=uint16),
 'parent': array([  nan,  997.,    8.,  343., 1065.,  354.,  370., 1069.]),
 'order': array([  0,   1, 639, 882, 935, 964, 998, 999], dtype=uint16)}

All parents along the tree are returned, in increasing order of `level` (0, 1, 2…), so the
highest, all-encompassing level comes first (`root` above).

- The fields contain all the parent regions, including the one passed in, which is last.
- The field `parent` gives the parent region id of each region in `id`: the same ids as in `id`,
  one level up.
- The field `order` holds values used for plotting; it is not the parent's index.

For example, the last `parent` is PPY, the closest parent of PPYd:

In [4]:
brain_regions.id2acronym(ancestors['parent'][-1])

array(['PPY'], dtype=object)

## Descendants

To find the descendants of a region, use `brain_regions.descendants`. Here with the region PPY:

In [5]:
brain_regions.descendants(ids=brain_regions.acronym2id('PPY'))

{'id': array([1069,  185,  193]),
 'name': array(['Parapyramidal nucleus', 'Parapyramidal nucleus deep part',
        'Parapyramidal nucleus superficial part'], dtype=object),
 'acronym': array(['PPY', 'PPYd', 'PPYs'], dtype=object),
 'rgb': array([[255, 179, 217],
        [255, 179, 217],
        [255, 179, 217]], dtype=uint8),
 'level': array([6, 7, 7], dtype=uint16),
 'parent': array([ 370., 1069., 1069.]),
 'order': array([ 998,  999, 1000], dtype=uint16)}

- The fields contain all the descendant regions, including the one passed in, which is first.
- The field `parent` gives the parent region id of each region in `id`.
- The field `order` holds values used for plotting; it is not the parent's index.

`descendants` returns the descendants from every branch below the region, for example for PTLp:

In [6]:
atlas_id = brain_regions.acronym2id('PTLp')
# Print the acronyms of the descendants of this region
print(brain_regions.descendants(ids=atlas_id)['acronym'])
# Print the levels of the descendants of this region
print(brain_regions.descendants(ids=atlas_id)['level'])

['PTLp' 'PTLp1' 'PTLp2/3' 'PTLp4' 'PTLp5' 'PTLp6a' 'PTLp6b' 'VISa' 'VISa1'
 'VISa2/3' 'VISa4' 'VISa5' 'VISa6a' 'VISa6b' 'VISrl' 'VISrl1' 'VISrl2/3'
 'VISrl4' 'VISrl5' 'VISrl6a' 'VISrl6b']
[6 7 7 7 7 7 7 7 8 8 8 8 8 8 7 8 8 8 8 8 8]


## Leaf nodes

A leaf node is a region with no descendants. One way to check is that `descendants` returns only
the region itself. PPYd is a leaf node:

In [7]:
ppyd_id = brain_regions.acronym2id('PPYd')
len(brain_regions.descendants(ids=ppyd_id)['id']) == 1

True

`brain_regions.leaves` returns all the leaf nodes at once, which is faster:

In [8]:
brain_regions.leaves()

{'id': array([-614454277, -607344862, -607344858, ...,  607344858,  607344862,
         614454277], shape=(2077,)),
 'name': array(['Supraoculomotor periaqueductal gray (left)',
        'Interpeduncular nucleus rostrolateral (left)',
        'Interpeduncular nucleus dorsolateral (left)', ...,
        'Interpeduncular nucleus dorsolateral',
        'Interpeduncular nucleus rostrolateral',
        'Supraoculomotor periaqueductal gray'], shape=(2077,), dtype=object),
 'acronym': array(['Su3', 'IPRL', 'IPDL', ..., 'IPDL', 'IPRL', 'Su3'],
       shape=(2077,), dtype=object),
 'rgb': array([[255, 144, 255],
        [255, 166, 255],
        [255, 166, 255],
        ...,
        [255, 166, 255],
        [255, 166, 255],
        [255, 144, 255]], shape=(2077, 3), dtype=uint8),
 'level': array([6, 7, 7, ..., 7, 7, 6], shape=(2077,), dtype=uint16),
 'parent': array([-795., -100., -100., ...,  100.,  100.,  795.], shape=(2077,)),
 'order': array([842, 878, 877, ..., 877, 878, 842], shape=(2077,), 

So to check whether a region is a leaf node:

In [9]:
ppyd_id in brain_regions.leaves().id

True

## Regions at a given level

`brain_regions.level` holds the level of each region in the tree:

In [10]:
print(f'brain_regions.level contains {brain_regions.level.size} values, which are either {np.unique(brain_regions.level)}')

brain_regions.level contains 2655 values, which are either [ 0  1  2  3  4  5  6  7  8  9 10]


In [11]:
# The regions at level 0, the top of the tree
brain_regions.acronym[brain_regions.level == 0]  # root appears twice, once per hemisphere

array(['void', 'root', 'root'], dtype=object)